# Using Jev and Oracle AI Database to govern agent memory

Companion notebook for the Oracle AI Database article [*Using Jev and Oracle AI Database to govern agent memory*](https://blogs.oracle.com/developers/using-jev-and-oracle-ai-database-to-govern-agent-memory). Every example runs against a real Oracle AI Database 26ai and the TypeSafe API, with the outputs from one run saved in place.

Oracle stores canonical memory and enforces the tenant boundary. Jev assesses the evidence for a retrieval route, context selection, or a proposed fact. Application policy decides which results enter context and which candidates become durable memory.

The examples build on the published [multi-tenant memory schema](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/multitenant_schema_walkthrough.ipynb), [two-layer memory pattern](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/two_layer_pattern_walkthrough.ipynb), and [hybrid retrieval pipeline](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/hybrid_retrieval_pipeline.ipynb).

All customer data is synthetic. A full run creates new Acme and Globex fixture tenants in the `DB_USER` schema and normally makes 19 paid Jev calls, including the batching comparison. Model decisions can change that count. The notebook sends the synthetic evidence to TypeSafe; it doesn't issue refunds or generate customer responses.

## Prerequisites

- Oracle AI Database 26ai with Oracle Text at `localhost:1521/FREEPDB1` (or set `DB_DSN`).
- A user with `DB_DEVELOPER_ROLE`, `CREATE SESSION`, `CREATE TABLE`, `CREATE MINING MODEL`,
  `EXECUTE ON DBMS_RLS`.
  This is the same user the other memory notebooks use. Every object this notebook creates
  has a `jev_` prefix, so it doesn't collide with their `memory_ctx`, `set_memory_ctx`, or
  memory tables.
- Outbound HTTPS from the notebook to Oracle Object Storage. If the user doesn't already have
  the augmented `ALL_MINILM_L12_V2` embedding model (384-dim), §1a downloads and loads it.
- `python-oracledb` 2.5 or newer in Thin mode (the default), which supports the Developer Hub program identifier.
- A [TypeSafe API key](https://docs.typesafe.ai/).
- A `.env` with `DB_USER`, `DB_PASSWORD`, `DB_DSN`, and `TYPESAFE_API_KEY` at the repository
  root. From the repository root, open the notebook with
  `jupyter lab notebooks/oracle_jev_memory/oracle_jev_memory.ipynb`. Jupyter starts the kernel in
  the notebook's folder, so `load_dotenv()` walks up to the `.env` and the notebook can import
  `memory_examples.py`.
- Python 3.11 or newer. The first code cell installs the pinned Python packages into the
  running kernel.

Run the cells in order. Restart the kernel before a full rerun. Each run creates its own fixture tenants; the cleanup cell at the end removes them.

## 1. Environment & connection

In [1]:
# Pinned versions. Restart the kernel if pip replaces an already-imported package.
# typesafe-sdk needs h11 0.16 or newer, which httpcore releases before 1.0.9 reject. The
# httpcore floor keeps the JupyterLab environment's own httpx consistent after the install.
%pip install -q --disable-pip-version-check oracledb==26.0.0 typesafe-sdk==0.7.1 python-dotenv==1.2.3 "httpcore>=1.0.9"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import statistics
import time
from pathlib import Path

import oracledb
from dotenv import load_dotenv
from typesafe_sdk import Choice, Noul, Score

# Identify this notebook in Oracle's client metadata before creating a connection.
oracledb.defaults.program = "devrel-developerhub-using-jev-and-oracle-ai-database-to-govern-agent-memory"

# Jupyter starts the kernel in the notebook's folder, and load_dotenv() searches upward
# from there to find the repository .env.
load_dotenv()

if not os.getenv("DB_USER"):
    raise RuntimeError(
        "No DB_USER in the environment. Add DB_USER, DB_PASSWORD, DB_DSN, and "
        "TYPESAFE_API_KEY to the .env at the repository root, then open the notebook with "
        "jupyter lab notebooks/oracle_jev_memory/oracle_jev_memory.ipynb."
    )

DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_DSN = os.getenv("DB_DSN", "localhost:1521/FREEPDB1")

if not (Path.cwd() / "memory_examples.py").is_file():
    raise RuntimeError(
        "memory_examples.py isn't in the kernel's working directory. Open the notebook with "
        "jupyter lab notebooks/oracle_jev_memory/oracle_jev_memory.ipynb; see README.md."
    )
from memory_examples import MemoryDemo, route_policy, select_context, promotion_policy

demo = MemoryDemo(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN, program=oracledb.defaults.program)
print("Oracle:", demo.conn.version)
print("Thin mode:", demo.conn.thin)
demo.cur.execute("SELECT SYS_CONTEXT('USERENV', 'CLIENT_PROGRAM_NAME') FROM dual")
print("Connection program:", demo.cur.fetchone()[0])
print("Jev:", demo.model)

Oracle: 23.26.3.0.0
Thin mode: True
Connection program: devrel-developerhub-using-jev-and-oracle-ai-database-to-govern-agent-memory
Jev: jev-1.13.0


### 1a. Load the embedding model

Oracle generates every embedding in this notebook inside the database with `VECTOR_EMBEDDING`, using Oracle's augmented `all-MiniLM-L12-v2` ONNX model. "Augmented" means tokenization is part of the model graph, so the function takes plain text and returns a 384-dimensional vector.

If the connected user doesn't have the model yet, this cell downloads it from Oracle Object Storage (about 130 MB) and passes the bytes to the BLOB overload of `DBMS_VECTOR.LOAD_ONNX_MODEL`. The file never has to be on the database host, so no directory object or administrator step is needed. If the model is already loaded, the cell only checks its dimensions.

In [3]:
print(demo.ensure_embedding_model())

{'model': 'ALL_MINILM_L12_V2', 'status': 'already loaded', 'dimensions': 384}


### 1b. Create the schema objects and seed fixtures

`setup()` creates any missing `jev_` tables, the `jev_memory_ctx` application context, and the VPD policies. `seed()` then writes this run's synthetic Acme and Globex fixtures.

In [4]:
demo.setup()
print(demo.seed())

# Display helpers only: neither function calls Jev or changes its answers.
def show_request(label, state, questions):
    print(f"\n{label} - request")
    print(json.dumps({"model": demo.model, "state": state,
                  "questions": {k: q.model_dump(mode="json") for k, q in questions.items()}}, indent=2, ensure_ascii=False))

def show_response(response, elapsed_ms):
    print(f"Jev response | {elapsed_ms:.2f} ms | {response.usage.input_tokens} input tokens")
    print(json.dumps(response.model_dump(mode="json"), indent=2, ensure_ascii=False))

{'tenant': 'acme_482924ca75a8', 'entity_fixtures': 9, 'embedding_dimensions': 384}


## 2. Store typed and scoped memory

The typed DDL in [schema.sql](schema.sql) adapts the multi-tenant schema linked above. This subset includes `jev_conversation_memory`, `jev_entity_memory`, `jev_guideline_memory`, `jev_persona_memory`, and `jev_summarization_memory`. The examples exercise conversations and entities; the summary table remains available but unpopulated.

Every memory retains tenant scope. Nullable user, agent, and thread scope inherit within that tenant. VPD enforces tenancy using `jev_memory_ctx`, the application context dedicated to this demo. The demo sets identity explicitly; a deployed service must obtain it from trusted authentication middleware.

Conversation events provide provenance. Entity content is canonical; its 384-dimensional MiniLM embedding is derived and rebuildable. Supersession preserves the earlier version. Guideline and persona records are loaded by exact scope matching, outside ranked retrieval.

The extra `jev_assessment_memory` table records the evidence supplied to Jev, question definitions, and returned distributions with their policy and model versions. It is also tenant-protected.

In [5]:
checks = demo.database_checks()
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(checks.values()), checks

PASS  unfiltered_select_tenant_isolated
PASS  missing_context_fails_closed
PASS  cross_tenant_insert_blocked
PASS  scope_and_lifecycle_before_ranking
PASS  both_hybrid_pools_used
PASS  supersession_preserves_history
PASS  embedding_rebuilt_from_canonical
PASS  stale_evidence_blocks_promotion
PASS  failed_promotion_leaves_no_memory
PASS  cross_tenant_source_hidden


## 3. Choose a retrieval route

Jev chooses a route from supplied options. Each option maps to application-owned queries; the model never supplies SQL or tenant identity. Low confidence goes to clarification. The current refund guideline and user's preferences are mandatory for this refund workflow regardless of the chosen route.

For prior-resolution requests, Oracle filters scope and validity before generating vector and Oracle Text candidate pools. Reciprocal rank fusion adds `1 / (60 + rank)` for each pool containing a row. The small fixture uses exact vector distances, so an approximate vector index is unnecessary. Oracle generates the MiniLM embeddings in the database.

The code below constructs a `Choice`, calls `demo.client.system_one(...)`, and displays the complete SDK response before applying the routing policy. `record_assessment` only saves that response to Oracle; it makes no additional API call.

In [6]:
stage_started = time.perf_counter()
requests = [
    ("Can you do what you did last time? I need a refund for TX-200, bought 45 days ago.", "prior_resolution"),
    ("What is the current refund policy?", "current_policy"),
    ("Can you help me with that thing?", "clarify"),
]
questions = {
    "route": Choice(
        instructions="Which evidence route does `request` need? Choose clarification when the subject cannot be identified.",
        criteria={
            "prior_resolution": "The user refers to a previous resolution or asks to repeat it.",
            "current_policy": "The user asks about applicable refund rules without reference to a prior resolution.",
            "clarify": "The subject or requested action is not identifiable.",
        },
    )
}
routes = []
for message, expected in requests:
    state = {"request": message}
    show_request("Retrieval routing", state, questions)

    started = time.perf_counter()
    response = demo.client.system_one(model=demo.model, state=state, questions=questions)
    elapsed_ms = (time.perf_counter() - started) * 1000

    show_response(response, elapsed_ms)
    answers = demo.record_assessment("routing", state, questions, response, elapsed_ms)
    route = route_policy(answers["route"])
    print("Application route:", route, "| Fixture expectation:", expected)
    routes.append({"request": message, "expected": expected, "actual": route, "answer": answers["route"]})

# Database operations remain outside the model's control.
demo.request = requests[0][0]
demo.mandatory = demo.mandatory_context()
demo.candidates = (
    demo.hybrid_retrieve("Acme previous refund exception transaction approval", "refund OR approval")
    if routes[0]["actual"] == "prior_resolution" else []
)
retrieval = {"routes": routes, "mandatory": demo.mandatory, "candidates": demo.candidates}
demo.results["retrieval"] = retrieval
demo.results.setdefault("stage_wall_ms", {})["retrieval"] = round((time.perf_counter()-stage_started)*1000, 2)
print("Mandatory records:", len(demo.mandatory))
for row in demo.candidates:
    print(row["key"], "vector rank:", row["vector_rank"], "lexical rank:", row["lexical_rank"],
          "RRF:", round(row["rrf"], 5), "|", row["content"])



Retrieval routing - request
{
  "model": "jev-1.13.0",
  "state": {
    "request": "Can you do what you did last time? I need a refund for TX-200, bought 45 days ago."
  },
  "questions": {
    "route": {
      "type": "choice",
      "instructions": "Which evidence route does `request` need? Choose clarification when the subject cannot be identified.",
      "criteria": {
        "prior_resolution": "The user refers to a previous resolution or asks to repeat it.",
        "current_policy": "The user asks about applicable refund rules without reference to a prior resolution.",
        "clarify": "The subject or requested action is not identifiable."
      }
    }
  }
}
Jev response | 278.54 ms | 399 input tokens
{
  "model": "jev-1.13.0",
  "usage": {
    "input_tokens": 399,
    "output_tokens": 42
  },
  "answers": {
    "route": {
      "type": "choice",
      "choice": "prior_resolution",
      "confidence": 0.91,
      "probabilities": {
        "prior_resolution": 0.94,
        

## 4. Select evidence for context

Each candidate receives an applicability classification and a separate relevance score. The previous refund is useful history, with its transaction limitation intact. A relevant passage does not become approval for a new refund.

The application reserves mandatory evidence before adding optional memories. This example enforces a UTF-8 byte budget over individual serialized evidence items. The count excludes separators and any surrounding request structure. A deployed context builder needs the chosen reasoning model's tokenizer and room for prompt formatting.

In [7]:
stage_started = time.perf_counter()
state = {
    "request": demo.request,
    "guidelines": demo.guideline,
    "candidates": {row["key"]: row for row in demo.candidates},
}
questions = {}
for row in demo.candidates:
    key = row["key"]
    questions[key + "_use"] = Choice(
        instructions=f"How may `candidates.{key}.content` be used for `request`, given `guidelines`? Historical exceptions do not authorize a new transaction.",
        criteria={
            "applicable": "Directly applicable current evidence for this task.",
            "history": "Relevant history only; must preserve its original transaction or scope limitation.",
            "irrelevant": "Does not help this request.",
            "insufficient": "Not enough evidence to determine how this applies.",
        },
    )
    questions[key + "_relevance"] = Score(
        instructions=f"How directly does `candidates.{key}.content` help answer `request`?",
        criteria=["Unrelated to the request.", "Provides useful background.", "Directly addresses the request."],
    )

if questions:
    show_request("Context selection: all candidate questions in one call", state, questions)

    started = time.perf_counter()
    response = demo.client.system_one(model=demo.model, state=state, questions=questions)
    elapsed_ms = (time.perf_counter() - started) * 1000

    show_response(response, elapsed_ms)
    answers = demo.record_assessment("selection", state, questions, response, elapsed_ms)
    selection = select_context(demo.candidates, answers, demo.mandatory)
    selection["answers"] = answers
else:
    selection = {"evidence": demo.mandatory, "decisions": [], "status": "No optional candidates; no Jev call."}

demo.results["selection"] = selection
demo.results["stage_wall_ms"]["selection"] = round((time.perf_counter()-stage_started)*1000, 2)
print("Application selection decisions:")
print(json.dumps(selection["decisions"], indent=2, ensure_ascii=False))
print("Evidence package for the reasoning model (no generation call is made):")
print(json.dumps(selection["evidence"], indent=2, ensure_ascii=False))



Context selection: all candidate questions in one call - request
{
  "model": "jev-1.13.0",
  "state": {
    "request": "Can you do what you did last time? I need a refund for TX-200, bought 45 days ago.",
    "guidelines": "Refunds within 30 days follow standard policy. Older purchases require a new approval for the specific transaction. Prior exceptions confer no future entitlement.",
    "candidates": {
      "c0": {
        "id": "482924ca75a8_exception",
        "version": 1,
        "source_event_id": "482924ca75a8_approval",
        "content": "Acme received a one-time refund for TX-100 at 45 days. Approval was limited to TX-100 and grants no future entitlement.",
        "rrf": 0.03278688524590164,
        "vector_rank": 1,
        "lexical_rank": 1,
        "key": "c0"
      },
      "c1": {
        "id": "482924ca75a8_current_process",
        "version": 2,
        "source_event_id": "482924ca75a8_new_source",
        "content": "Acme refund requests must use the support por

## 5. Evaluate promotion before writing a fact

A broad refund entitlement and an invented transaction approval should fail. A narrow account observation tied to the original approval can be promoted. Missing sources and exact duplicates are handled without a model call.

Support, scope, and conflict remain separate assessments. The thresholds here are illustrative rather than calibrated. An accepted candidate is written with its source event and embedding in one transaction, along with a promotion event. The write path locks and rechecks the specific evidence used in the assessment. The database check above also simulates stale evidence and verifies that it leaves no promoted row.

In [8]:
stage_started = time.perf_counter()
promotion_questions = {
    "supported": Noul(instructions="Is every factual claim in `candidate` explicitly supported by `source.content`? Do not infer future rights from a past event."),
    "scope_fits": Noul(instructions="Does `candidate` preserve all limitations in `source.content`, including the customer, transaction, and one-time nature?"),
    "conflict": Noul(instructions="Does `candidate` contradict `guideline` or the current facts in `existing`? A scoped historical exception does not contradict the requirement for new approval."),
}
narrow = "On 2026-08-22, Ada approved a one-time refund for Acme transaction TX-100, which was 45 days old; this does not authorize future refunds."
cases = [
    ("broad", "Acme is always eligible for refunds after the standard window.", demo.source_id),
    ("invented", "Ada approved a refund for Acme transaction TX-200.", demo.source_id),
    ("narrow", narrow, demo.source_id),
    ("duplicate", narrow, demo.source_id),
    ("missing", "Acme has unlimited refunds.", demo.run_id + "_missing"),
]
promotions = []
for label, candidate, source_id in cases:
    # Oracle reads, source validation, and exact deduplication; no API call here.
    prepared = demo.prepare_promotion(candidate, source_id)
    if "decision" in prepared:
        result = dict(label=label, **prepared)
        print(f"\n{label}: Jev skipped - {result['decision']}")
    else:
        state = prepared["state"]
        show_request(f"Promotion: {label}", state, promotion_questions)

        started = time.perf_counter()
        response = demo.client.system_one(model=demo.model, state=state, questions=promotion_questions)
        elapsed_ms = (time.perf_counter() - started) * 1000

        show_response(response, elapsed_ms)
        answers = demo.record_assessment("promotion_" + label, state, promotion_questions, response, elapsed_ms)
        decision = promotion_policy(answers)
        print("Application promotion decision:", decision)
        result = {"label": label, "candidate": candidate, "decision": decision, "answers": answers, "jev_called": True}
        if decision == "promote":
            result["id"] = demo.commit_promotion(
                label, candidate, prepared["source"], prepared["current"], prepared["mandatory_snapshot"]
            )
    promotions.append(result)

demo.results["promotion"] = promotions
demo.results["stage_wall_ms"]["promotion"] = round((time.perf_counter()-stage_started)*1000, 2)
demo.cur.execute("SELECT id, version, source_event_id, content FROM jev_entity_memory WHERE written_by = 'promotion_job' AND id LIKE :prefix",
                 prefix=demo.run_id + "_promoted_%")
print("Promoted canonical rows:")
for row in demo.cur:
    print(row)



Promotion: broad - request
{
  "model": "jev-1.13.0",
  "state": {
    "candidate": "Acme is always eligible for refunds after the standard window.",
    "source": {
      "event_id": "482924ca75a8_approval",
      "content": "On 2026-08-22, manager Ada approved a one-time refund for customer Acme, transaction TX-100 only, despite its being 45 days old. This approval does not authorize any other transaction or future refunds."
    },
    "guideline": [
      {
        "id": "482924ca75a8_policy",
        "version": 1,
        "source_event_id": "482924ca75a8_policy_source",
        "content": "Refunds within 30 days follow standard policy. Older purchases require a new approval for the specific transaction. Prior exceptions confer no future entitlement."
      },
      {
        "id": "482924ca75a8_persona",
        "source_event_id": "482924ca75a8_preference",
        "content": "Use concise explanations with source references."
      }
    ],
    "existing": [
      {
        "id": 

## 6. Compare batched questions with separate calls

The comparison sends the same state and questions as one request or as sequential requests. Trial order alternates to reduce order effects. Reported latency includes SDK/network time and any retries; it excludes the local audit insert. Both modes reuse the same client connection. There is no reasoning-model baseline.

The estimate uses [TypeSafe's published rate for Jev 1.13](https://docs.typesafe.ai/models), checked September 23, 2026: $0.042 per million input tokens, with free outputs. Update `INPUT_PRICE_PER_MILLION` in `memory_examples.py` if the rate changes. Actual usage comes from the API response. This small test can show local overhead and token duplication; it cannot establish production latency percentiles or decision accuracy.

In [9]:
state = {
    "candidate": "Ada approved TX-100 only; no future refunds are authorized.",
    "source": {"content": demo.source_text},
    "guideline": demo.guideline,
    "existing": [],
}
questions = promotion_questions
show_request("Batching comparison: identical evidence and questions in both modes", state, questions)
repeats = 3
first_call = len(demo.calls)
trials = []
for trial in range(repeats):
    modes = ("batch", "separate") if trial % 2 == 0 else ("separate", "batch")
    collected = {}
    for mode in modes:
        before = len(demo.calls)
        if mode == "batch":
            started = time.perf_counter()
            response = demo.client.system_one(model=demo.model, state=state, questions=questions)
            elapsed_ms = (time.perf_counter() - started) * 1000

            print(f"\nTrial {trial + 1}: batched questions")
            show_response(response, elapsed_ms)
            answers = demo.record_assessment("benchmark_batch", state, questions, response, elapsed_ms)
        else:
            answers = {}
            for key, question in questions.items():
                started = time.perf_counter()
                response = demo.client.system_one(model=demo.model, state=state, questions={key: question})
                elapsed_ms = (time.perf_counter() - started) * 1000

                print(f"\nTrial {trial + 1}: separate question '{key}'")
                show_response(response, elapsed_ms)
                answers.update(demo.record_assessment("benchmark_separate", state, {key: question}, response, elapsed_ms))
        calls = demo.calls[before:]
        collected[mode] = {
            "elapsed_ms": sum(c["elapsed_ms"] for c in calls),
            "input_tokens": sum(c["usage"]["input_tokens"] for c in calls),
            "estimated_usd": sum(c["estimated_usd"] for c in calls),
            "decision": promotion_policy(answers),
            "answers": answers,
        }
    trials.append(collected)

benchmark = {
    "repeats": repeats, "calls": len(demo.calls) - first_call, "trials": trials,
    "batch_median_ms": statistics.median(t["batch"]["elapsed_ms"] for t in trials),
    "separate_median_ms": statistics.median(t["separate"]["elapsed_ms"] for t in trials),
    "batch_input_tokens": sum(t["batch"]["input_tokens"] for t in trials),
    "separate_input_tokens": sum(t["separate"]["input_tokens"] for t in trials),
    "policy_agreement": all(t["batch"]["decision"] == t["separate"]["decision"] for t in trials),
}
demo.results["benchmark"] = benchmark
print("Batching summary (API timing excludes display and Oracle audit writes):")
print(json.dumps({k: v for k, v in benchmark.items() if k != "trials"}, indent=2, ensure_ascii=False))



Batching comparison: identical evidence and questions in both modes - request
{
  "model": "jev-1.13.0",
  "state": {
    "candidate": "Ada approved TX-100 only; no future refunds are authorized.",
    "source": {
      "content": "On 2026-08-22, manager Ada approved a one-time refund for customer Acme, transaction TX-100 only, despite its being 45 days old. This approval does not authorize any other transaction or future refunds."
    },
    "guideline": "Refunds within 30 days follow standard policy. Older purchases require a new approval for the specific transaction. Prior exceptions confer no future entitlement.",
    "existing": []
  },
  "questions": {
    "supported": {
      "type": "noul",
      "instructions": "Is every factual claim in `candidate` explicitly supported by `source.content`? Do not infer future rights from a past event."
    },
    "scope_fits": {
      "type": "noul",
      "instructions": "Does `candidate` preserve all limitations in `source.content`, includ

## 7. Save observations and check expected outcomes

Expected labels describe the fixture. Differences appear as evaluation misses, so they remain visible even when the notebook completes. The JSON result preserves the response probabilities and usage measurements. Oracle retains the supplied evidence and question definitions for inspection.

The schema owner can administer its own policies, and the demo setter accepts an explicit tenant. These examples verify VPD behavior for ordinary queries. Production deployment needs a separate runtime principal with trusted identity binding. Concurrent promotions also require a coordinated write protocol.


In [10]:
expected = {"broad":"reject_unsupported", "invented":"reject_unsupported", "narrow":"promote",
            "duplicate":"duplicate", "missing":"reject_missing_source"}
evaluation = {
    "routes": [{"expected":r["expected"], "actual":r["actual"], "match":r["expected"]==r["actual"]}
               for r in retrieval["routes"]],
    "promotion": [{"label":r["label"], "expected":expected[r["label"]], "actual":r["decision"],
                   "match":expected[r["label"]]==r["decision"]} for r in promotions],
    "exception_kept_as_history": any(r["id"]==demo.ids["exception"] and r["use"]=="history"
                                      for r in selection["decisions"]),
    "irrelevant_excluded": any(r["id"]==demo.ids["irrelevant"] and r["use"]=="exclude"
                                for r in selection["decisions"]),
}
demo.results["evaluation"] = evaluation
print(json.dumps(evaluation, indent=2))
result_path = demo.save()
print("Result file:", result_path.name)
print("API totals:", demo.results["totals"])


{
  "routes": [
    {
      "expected": "prior_resolution",
      "actual": "prior_resolution",
      "match": true
    },
    {
      "expected": "current_policy",
      "actual": "current_policy",
      "match": true
    },
    {
      "expected": "clarify",
      "actual": "clarify",
      "match": true
    }
  ],
  "promotion": [
    {
      "label": "broad",
      "expected": "reject_unsupported",
      "actual": "reject_unsupported",
      "match": true
    },
    {
      "label": "invented",
      "expected": "reject_unsupported",
      "actual": "reject_unsupported",
      "match": true
    },
    {
      "label": "narrow",
      "expected": "promote",
      "actual": "promote",
      "match": true
    },
    {
      "label": "duplicate",
      "expected": "duplicate",
      "actual": "duplicate",
      "match": true
    },
    {
      "label": "missing",
      "expected": "reject_missing_source",
      "actual": "reject_missing_source",
      "match": true
    }
  ],
  "except

## 8. Cleanup

Remove this run's fixture tenants, assessment records, and promoted rows, then close the connection. Skip this cell to keep the seeded data around for inspection. Rows from other runs are never touched.


In [11]:
print(demo.cleanup())
demo.close()


{'jev_guideline_memory': 1, 'jev_persona_memory': 1, 'jev_entity_memory': 10, 'jev_conversation_memory': 12, 'jev_summarization_memory': 0, 'jev_assessment_memory': 19}


## Further reading

- [Hybrid retrieval for agent memory](https://blogs.oracle.com/developers/hybrid-retrieval-for-agent-memory-vector-lexical-and-metadata-together)
- [TypeSafe primitives](https://docs.typesafe.ai/primitives)
- [Oracle VECTOR_EMBEDDING](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/vector_embedding.html)
